# 데이터 전처리

> **자료 출처** — 본 노트북은 기상청에서 개방한 「종관기상관측(ASOS) 일자료」(서울 108, 대구 143, 추풍령 135)와
> 「관측지점정보」를 이용하였으며, codysseus42가 결측·이상치 처리 및 지표 계산을 거쳐 재가공하였습니다.
> 원자료는 기상자료개방포털(data.kma.go.kr)에서 무료로 내려받을 수 있습니다.

## 작업 환경

In [1]:
import sys
import pandas as pd
import numpy as np
import glob
import platform
from IPython.display import display

python_version = sys.version_info
print(f"python version: {python_version}")
print(f"pandas version: {pd.__version__}")

python version: sys.version_info(major=3, minor=14, micro=7, releaselevel='final', serial=0)
pandas version: 3.0.6


## 데이터 로딩(서울)

원본데이터를 취합하여 1960-01-01~2026-09-13 일 까지의 24363 데이터를 확인히였다. 이후 작업을 위해서 컬럼명을 영어로 변경했다.

불쾌지수 산출과 습함과 온도 분석의 핵심이 되는 부분과 보조지표를 분리하였다. 
핵심
    "평균기온(°C)": "tavg", "최저기온(°C)": "tmin", "최고기온(°C)": "tmax",
    "평균 상대습도(%)": "rh", "평균 이슬점온도(°C)": "dew", "평균 증기압(hPa)": "vp",
보조
    "일강수량(mm)": "precip", "합계 일조시간(hr)": "sunshine", "평균 풍속(m/s)": "wind",

df_seoul_analysis.info()를 통해 확인한 결과 보조지표중에는 일강수량 결측 데이터가 크다는 부분이 눈에 띈다.
강수량이 없을 경우 NULL로 처리된 것으로 보인다.(이후 무강수일은 NAN, 미량 강수일은 0으로 처리되었을 가능성이 높다고 보고 확인 둘다 수치상 0으로 간주한다.)

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

#원본 데이터 파일 목록
print("====================== 원본 데이터 파일 목록 ======================")
files = sorted(glob.glob("data/raw/seoul/seoul_*.csv"))
display(files)

#데이터 구조
print("====================== 데이터 구조(seoul_1960.csv) ======================")
raw_seoul = pd.read_csv("data/raw/seoul/seoul_1960.csv", encoding="cp949")
print(f"기간: {raw_seoul['일시'].min()} ~ {raw_seoul['일시'].max()}")
raw_seoul.info()
print(f"컬럼 {len(raw_seoul.columns)}개")
print(f"행 {len(raw_seoul)}개")
print(list(raw_seoul.columns))
display(raw_seoul.head())


# 데이터 통합
print("====================== 데이터 통합 ======================")
df_seoul = pd.concat((pd.read_csv(f,encoding="cp949") for f in files), ignore_index=True)
print(f"기간: {df_seoul['일시'].min()} ~ {df_seoul['일시'].max()}")
df_seoul.info()
print(f"컬럼 {len(df_seoul.columns)}개")
print(f"행 {len(df_seoul)}개")
print(list(df_seoul.columns))

# 데이터 기초 전처리(컬럼 이름 변경, 결측 마스크 대상 컬럼 추출)
print("====================== 데이터 기초 전처리 ======================")
COLS = {
    # 식별
    "지점": "stn", "지점명": "stn_name", "일시": "date",
    # 핵심 — 결측 마스크 대상
    "평균기온(°C)": "tavg", "최저기온(°C)": "tmin", "최고기온(°C)": "tmax",
    "평균 상대습도(%)": "rh", "최소 상대습도(%)": "rh_min","평균 이슬점온도(°C)": "dew", "평균 증기압(hPa)": "vp",
    # 보조 — 마스크 제외
    "일강수량(mm)": "precip", "합계 일조시간(hr)": "sunshine", "평균 풍속(m/s)": "wind","최고기온 시각(hhmi)": "tmax_time","최소 상대습도 시각(hhmi)": "rh_min_time",
}
CORE = ["tavg", "tmin", "tmax", "rh","rh_min", "dew", "vp"]
AUX  = ["precip", "sunshine", "wind" ,"tmax_time", "rh_min_time"]

df_seoul_analysis = df_seoul.rename(columns=COLS)[list(COLS.values())].copy()
df_seoul_analysis["date"] = pd.to_datetime(df_seoul_analysis["date"])

print(f"원본 {len(df_seoul.columns)}컬럼 → 분석용 {len(df_seoul_analysis.columns)}컬럼")
display(pd.DataFrame(COLS.items(), columns=["원본", "분석용"]))

df_seoul_analysis.info()

display(df_seoul_analysis.head())
display(df_seoul_analysis.loc[12179:12183])
display(df_seoul_analysis.tail())


df_seoul_analysis[CORE].describe() 


====================== 원본 데이터 파일 목록 ======================


['data/raw/seoul/seoul_1960.csv',
 'data/raw/seoul/seoul_1970.csv',
 'data/raw/seoul/seoul_1980.csv',
 'data/raw/seoul/seoul_1990.csv',
 'data/raw/seoul/seoul_2000.csv',
 'data/raw/seoul/seoul_2010.csv',
 'data/raw/seoul/seoul_2020.csv']

====================== 데이터 구조(seoul_1960.csv) ======================
기간: 1960-01-01 ~ 1969-12-31
<class 'pandas.DataFrame'>
RangeIndex: 3653 entries, 0 to 3652
Data columns (total 62 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   지점                   3653 non-null   int64  
 1   지점명                  3653 non-null   str    
 2   일시                   3653 non-null   str    
 3   평균기온(°C)             3653 non-null   float64
 4   최저기온(°C)             3652 non-null   float64
 5   최저기온 시각(hhmi)        3499 non-null   float64
 6   최고기온(°C)             3652 non-null   float64
 7   최고기온 시각(hhmi)        3499 non-null   float64
 8   강수 계속시간(hr)          1189 non-null   float64
 9   10분 최다 강수량(mm)       1430 non-null   float64
 10  10분 최다강수량 시각(hhmi)   1043 non-null   float64
 11  1시간 최다강수량(mm)        1437 non-null   float64
 12  1시간 최다 강수량 시각(hhmi)  1073 non-null   float64
 13  일강수량(mm)             1556 non-null   float64
 14  최대

,지점,지점명,일시,평균기온(°C),최저기온(°C),최저기온 시각(hhmi),최고기온(°C),최고기온 시각(hhmi),강수 계속시간(hr),10분 최다 강수량(mm),10분 최다강수량 시각(hhmi),1시간 최다강수량(mm),1시간 최다 강수량 시각(hhmi),일강수량(mm),최대 순간 풍속(m/s),최대 순간 풍속 풍향(16방위),최대 순간풍속 시각(hhmi),최대 풍속(m/s),최대 풍속 풍향(16방위),최대 풍속 시각(hhmi),평균 풍속(m/s),풍정합(100m),최다풍향(16방위),평균 이슬점온도(°C),최소 상대습도(%),최소 상대습도 시각(hhmi),평균 상대습도(%),평균 증기압(hPa),평균 현지기압(hPa),최고 해면기압(hPa),최고 해면기압 시각(hhmi),최저 해면기압(hPa),최저 해면기압 시각(hhmi),평균 해면기압(hPa),가조시간(hr),합계 일조시간(hr),1시간 최다일사 시각(hhmi),1시간 최다일사량(MJ/m2),합계 일사량(MJ/m2),일 최심신적설(cm),일 최심신적설 시각(hhmi),일 최심적설(cm),일 최심적설 시각(hhmi),합계 3시간 신적설(cm),평균 전운량(1/10),평균 중하층운량(1/10),평균 지면온도(°C),최저 초상온도(°C),평균 5cm 지중온도(°C),평균 10cm 지중온도(°C),평균 20cm 지중온도(°C),평균 30cm 지중온도(°C),0.5m 지중온도(°C),1.0m 지중온도(°C),1.5m 지중온도(°C),3.0m 지중온도(°C),5.0m 지중온도(°C),합계 대형증발량(mm),합계 소형증발량(mm),9-9강수(mm),기사,안개 계속시간(hr)
0,108,서울,1960-01-01,-1.6,-5.2,728.0,2.2,1330.0,NaN,NaN,NaN,NaN,NaN,NaN,7.6,290.0,1213.0,4.7,270.0,1330.0,1.7,1469,NaN,NaN,50.0,NaN,68.3,3.7,1024.2,1025.9,NaN,1023.3,NaN,1024.2,NaN,6.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,-1.6,-8.6,-0.5,-0.4,0.1,0.7,2.5,5.7,11.2,14.2,15.7,NaN,0.8,0.4,{안개}{강도0}0001-{안개}{강도1}0605-{안개}{강도2}0615-{안개}...,NaN
1,108,서울,1960-01-02,-1.9,-5.6,707.0,1.2,1530.0,NaN,NaN,NaN,NaN,NaN,0.4,4.8,70.0,945.0,3.0,50.0,630.0,1.3,1123,NaN,NaN,64.0,NaN,87.7,4.7,1021.6,1025.3,NaN,1019.7,NaN,1021.6,NaN,0.0,NaN,NaN,NaN,2.8,622.0,2.8,622.0,NaN,8.0,NaN,-0.9,-9.3,-0.6,-0.4,0.1,0.7,2.5,5.6,10.9,14.1,15.6,NaN,0.6,0.0,{안개}{강도0}0720-{안개}{강도1}1254-{안개}{강도0}2045- {눈...,NaN
2,108,서울,1960-01-03,4.0,-2.1,34.0,8.7,1533.0,NaN,NaN,NaN,NaN,NaN,0.0,7.6,230.0,1545.0,6.7,230.0,1700.0,3.0,2592,NaN,NaN,72.0,NaN,81.3,6.8,1011.8,1019.7,NaN,1009.0,NaN,1011.8,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.3,NaN,1.4,-5.0,-0.2,-0.2,0.2,0.7,2.4,5.5,10.8,14.0,15.5,NaN,1.3,0.0,-{안개}{강도0}- {안개}{강도1}0650-{안개}{강도0}0946-1535. ...,NaN
3,108,서울,1960-01-04,7.5,1.2,2359.0,10.8,1330.0,NaN,NaN,NaN,NaN,NaN,0.0,13.9,230.0,1230.0,8.0,230.0,1400.0,4.4,3802,NaN,NaN,57.0,NaN,79.7,8.1,1007.4,1011.9,NaN,1005.6,NaN,1007.4,NaN,2.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.7,NaN,4.8,-1.8,0.5,0.0,0.3,0.8,2.4,5.4,10.6,14.0,15.5,NaN,2.0,NaN,-{안개}{강도0}-1350. {연무}{강도0}1351-1750. {안개}{강도...,NaN
4,108,서울,1960-01-05,-4.6,-8.2,2400.0,1.3,0.0,NaN,NaN,NaN,NaN,NaN,NaN,15.0,290.0,1432.0,9.8,290.0,1430.0,5.1,4406,NaN,NaN,36.0,NaN,44.0,1.8,1020.7,1026.7,NaN,1011.9,NaN,1020.7,NaN,8.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,-2.7,-10.0,-0.2,-0.1,0.2,0.7,2.4,5.4,10.5,13.9,15.5,NaN,1.9,NaN,{연무}{강도0}0845-1410.,NaN


====================== 데이터 통합 ======================
기간: 1960-01-01 ~ 2026-09-13
<class 'pandas.DataFrame'>
RangeIndex: 24363 entries, 0 to 24362
Data columns (total 62 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   지점                   24363 non-null  int64  
 1   지점명                  24363 non-null  str    
 2   일시                   24363 non-null  str    
 3   평균기온(°C)             24363 non-null  float64
 4   최저기온(°C)             24361 non-null  float64
 5   최저기온 시각(hhmi)        24208 non-null  float64
 6   최고기온(°C)             24361 non-null  float64
 7   최고기온 시각(hhmi)        24208 non-null  float64
 8   강수 계속시간(hr)          9347 non-null   float64
 9   10분 최다 강수량(mm)       6969 non-null   float64
 10  10분 최다강수량 시각(hhmi)   5127 non-null   float64
 11  1시간 최다강수량(mm)        6977 non-null   float64
 12  1시간 최다 강수량 시각(hhmi)  5192 non-null   float64
 13  일강수량(mm)             9718 non-null   float64
 14  최대 순간 풍속(m/s)   

...

<class 'pandas.DataFrame'>
RangeIndex: 24363 entries, 0 to 24362
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   stn          24363 non-null  int64         
 1   stn_name     24363 non-null  str           
 2   date         24363 non-null  datetime64[us]
 3   tavg         24363 non-null  float64       
 4   tmin         24361 non-null  float64       
 5   tmax         24361 non-null  float64       
 6   rh           24363 non-null  float64       
 7   rh_min       24361 non-null  float64       
 8   dew          23997 non-null  float64       
 9   vp           24363 non-null  float64       
 10  precip       9718 non-null   float64       
 11  sunshine     24240 non-null  float64       
 12  wind         24354 non-null  float64       
 13  tmax_time    24208 non-null  float64       
 14  rh_min_time  23540 non-null  float64       
dtypes: datetime64[us](1), float64(12), int64(1), str(1)
memory usage

,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time
0,108,서울,1960-01-01,-1.6,-5.2,2.2,68.3,50.0,NaN,3.7,NaN,6.7,1.7,1330.0,NaN
1,108,서울,1960-01-02,-1.9,-5.6,1.2,87.7,64.0,NaN,4.7,0.4,0.0,1.3,1530.0,NaN
2,108,서울,1960-01-03,4.0,-2.1,8.7,81.3,72.0,NaN,6.8,0.0,0.0,3.0,1533.0,NaN
3,108,서울,1960-01-04,7.5,1.2,10.8,79.7,57.0,NaN,8.1,0.0,2.6,4.4,1330.0,NaN
4,108,서울,1960-01-05,-4.6,-8.2,1.3,44.0,36.0,NaN,1.8,NaN,8.2,5.1,0.0,NaN


,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time
12179,108,서울,1993-05-06,19.4,15.9,24.3,72.8,52.0,14.2,16.2,NaN,3.5,1.2,1419.0,1354.0
12180,108,서울,1993-05-07,20.0,15.4,26.0,68.8,33.0,13.6,15.7,NaN,8.7,2.9,1220.0,1230.0
12181,108,서울,1993-05-08,17.0,13.8,22.2,61.8,30.0,9.1,11.6,NaN,8.5,3.0,1444.0,1300.0
12182,108,서울,1993-05-09,16.3,13.4,21.4,62.5,30.0,8.9,11.4,NaN,0.8,2.0,1620.0,1713.0
12183,108,서울,1993-05-10,17.3,12.1,24.3,59.8,29.0,8.9,11.4,NaN,10.9,3.2,1458.0,1612.0


,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time
24358,108,서울,2026-09-09,21.3,17.8,26.1,50.9,41.0,10.6,12.8,0.0,7.1,4.2,1347.0,1422.0
24359,108,서울,2026-09-10,18.4,15.8,21.5,57.5,47.0,9.7,12.0,NaN,0.2,2.3,1414.0,1220.0
24360,108,서울,2026-09-11,20.0,13.4,25.7,62.3,39.0,12.1,14.2,NaN,9.0,1.6,1345.0,1525.0
24361,108,서울,2026-09-12,20.5,17.1,23.6,68.4,55.0,14.3,16.3,0.0,0.3,1.5,1318.0,1539.0
24362,108,서울,2026-09-13,21.0,18.5,23.8,71.1,62.0,15.5,17.6,0.2,0.4,1.4,1521.0,1652.0


,tavg,tmin,tmax,rh,rh_min,dew,vp
count,24363.000000,24361.000000,24361.000000,24363.000000,24361.000000,23997.000000,24363.000000
mean,12.537409,8.570966,17.196047,65.196757,44.269804,5.566437,11.821467
std,10.519603,10.619165,10.730718,14.531331,16.514484,11.742035,8.232165
min,-16.400000,-20.200000,-13.600000,17.900000,7.000000,-27.600000,0.700000
25%,3.500000,-0.200000,8.000000,54.800000,32.000000,-3.500000,4.800000
50%,14.000000,9.300000,19.100000,65.500000,43.000000,6.300000,9.600000
75%,21.900000,18.000000,26.500000,75.800000,55.000000,15.800000,18.000000
max,33.700000,30.300000,39.600000,99.800000,97.000000,27.200000,36.200000


## 전처리(서울)
평균 이슬점온도(°C)가 1960년(366일)에 측정 되지 않았음을 확인 했다. 해당년도를 제외하고 1961년도 부터 분석 하기로 한다.
합계일조 시간도 1965년과 1966년에 눈에 띄는 결측치가 확인된다. 비핵심지표라 해당 지표를 활용 할 경우에만 배제하기로 한다.
최고기온 측정시간은 1960년에, 최저습도 결측시간은 1960~1962년에 걸친 결측치가 눈에 띄나 1960년은 제외 되었고 비핵심 지표임으로 사용시에만 배제하기로 한다.
결측치 해석의 타당성에 대한 AI의 지적에 따라 확인한 결과 일강수량과 일조시간에 0값과 결측이 혼재되었으며 양적 관계에서 이들의 처리에 차이가 있는 것을 확인했다.
일조시간은 0이 관측되는 날의 비율이 유의미하게 존재하는 반면 결측일은 극히적어서 실제 결측치로 간주 함이 타당해 보인다.(일조시간 결측 123건 중 87%가 강수일이라는 지적이 사실로 확인됨에 따라 이 역시 날씨의 영향일 수 있으나 양이 비교적 적고, 비핵심 지표여서 해당 분석에서만 배제하기로 한다.)
반면 일강수량은 전체기간중 60.11%에  달하는 기간이 결측처리 되어있어 무강수일로 추정되고 0일은 기상청의 규정에 따라 미량 강수일가능성이 있다. 결측치와 0값에 대한 확실한 기준을 발견하지 못하였으나 둘다 수치상에서 0으로 간주함이 합당하다. 따라서 일강수량에 결측치는 없다고 친다.(한계)
평균풍속의 미량의 결측치(9일) 역시 극소수의 결측치로 0이 공존하지 않는 특성상 실제 결측치로 간주하여 해당 지표를 활용한 분석에서는 제외한다.
핵심 지표중 결측이 있는 최대,최저 기온, 최저 상대습도의 3일 결칙치는 실제 결측치이며 분석에서 배제한다.(공통:1967-02-19,최고 기온, 최저 상대습도:2017-10-12,최저 기온:2022-08-08)


In [3]:
print("====================== 결측치 연간분포 ======================")
na = df_seoul_analysis.set_index("date")[CORE + AUX].isna()
display(na.resample("YE").sum().loc[lambda d: d.sum(axis=1) > 0])

print("====================== 소수 결측치 ======================")
print("====================== 평균 풍속 결측치 ======================")
nc = df_seoul_analysis[df_seoul_analysis["wind"].isna()]
nan_wind = len(nc)
nc = df_seoul_analysis[df_seoul_analysis["wind"]==0.0]
no_wind = len(nc)
print(f"풍속 결측치: {nan_wind}개, 0 값: {no_wind}개,NAN 비율: {nan_wind/(len(df_seoul_analysis))*100:.2f}%, 0값 비율: {no_wind/(len(df_seoul_analysis)-nan_wind)*100:.2f}%")
print(len(nc)/len(df_seoul_analysis))
display(df_seoul_analysis.loc[df_seoul_analysis["wind"].isna()][["stn","stn_name","date", "wind"]])
print("====================== 최저기온 결측치 ======================")
display(df_seoul_analysis.loc[df_seoul_analysis["tmin"].isna()][["stn","stn_name","date", "tmin"]])
print("====================== 최고기온 결측치 ======================")
display(df_seoul_analysis.loc[df_seoul_analysis["tmax"].isna()][["stn","stn_name","date", "tmax"]])
print("====================== 최저 상대습도 결측치 ======================")
display(df_seoul_analysis.loc[df_seoul_analysis["rh_min"].isna()][["stn","stn_name","date", "rh_min"]])

====================== 결측치 연간분포 ======================


,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time
date,,,,,,,,,,,,
1960-12-31,0,0,0,0,0,366,0,215,0,0,153,366
1961-12-31,0,0,0,0,0,0,0,198,0,0,0,365
1962-12-31,0,0,0,0,0,0,0,214,0,0,0,90
1963-12-31,0,0,0,0,0,0,0,202,0,0,0,0
1964-12-31,0,0,0,0,0,0,0,207,0,0,0,0
1965-12-31,0,0,0,0,0,0,0,216,42,0,0,0
1966-12-31,0,0,0,0,0,0,0,209,56,0,0,0
1967-12-31,0,1,1,0,1,0,0,212,2,0,1,1
1968-12-31,0,0,0,0,0,0,0,228,2,0,0,0


====================== 소수 결측치 ======================
====================== 평균 풍속 결측치 ======================
풍속 결측치: 9개, 0 값: 0개,NAN 비율: 0.04%, 0값 비율: 0.00%
0.0


,stn,stn_name,date,wind
8597,108,서울,1983-07-16,NaN
21106,108,서울,2017-10-14,NaN
21158,108,서울,2017-12-05,NaN
21159,108,서울,2017-12-06,NaN
23354,108,서울,2023-12-10,NaN
23355,108,서울,2023-12-11,NaN
23446,108,서울,2024-03-11,NaN
23447,108,서울,2024-03-12,NaN
23449,108,서울,2024-03-14,NaN


====================== 최저기온 결측치 ======================


,stn,stn_name,date,tmin
2606,108,서울,1967-02-19,NaN
22865,108,서울,2022-08-08,NaN


====================== 최고기온 결측치 ======================


,stn,stn_name,date,tmax
2606,108,서울,1967-02-19,NaN
21104,108,서울,2017-10-12,NaN


====================== 최저 상대습도 결측치 ======================


,stn,stn_name,date,rh_min
2606,108,서울,1967-02-19,NaN
21104,108,서울,2017-10-12,NaN


In [4]:
print("====================== 합계 일조시간 결측치 확인 ======================")
nb = df_seoul_analysis[(df_seoul_analysis["date"] >= "1965-01-01") & (df_seoul_analysis["date"] < "1966-12-31")].set_index("date")[CORE + AUX].isna()
display(nb.resample("ME").sum().loc[lambda d: d.sum(axis=1) > 0])

====================== 합계 일조시간 결측치 확인 ======================


,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time
date,,,,,,,,,,,,
1965-01-31,0,0,0,0,0,0,0,18,0,0,0,0
1965-02-28,0,0,0,0,0,0,0,20,1,0,0,0
1965-03-31,0,0,0,0,0,0,0,24,3,0,0,0
1965-04-30,0,0,0,0,0,0,0,22,3,0,0,0
1965-05-31,0,0,0,0,0,0,0,18,0,0,0,0
1965-06-30,0,0,0,0,0,0,0,20,2,0,0,0
1965-07-31,0,0,0,0,0,0,0,9,11,0,0,0
1965-08-31,0,0,0,0,0,0,0,9,13,0,0,0
1965-09-30,0,0,0,0,0,0,0,22,0,0,0,0


In [5]:
print("====================== 합계 일조시간 결측치,0 값 비율 확인 ======================")
nc = df_seoul_analysis[df_seoul_analysis["sunshine"].isna()]
nan_sunshine = len(nc)
nc = df_seoul_analysis[df_seoul_analysis["sunshine"]==0.0]
no_sunshine = len(nc)
print(f"합계 일조시간 결측치: {nan_sunshine}개, 0 값: {no_sunshine}개,NAN 비율: {nan_sunshine/(len(df_seoul_analysis))*100:.2f}%, 0값 비율: {no_sunshine/(len(df_seoul_analysis)-nan_sunshine)*100:.2f}%")
print(len(nc))
print("====================== 일강수량 NAN, 0 값 비율 확인 ======================")
nc = df_seoul_analysis[df_seoul_analysis["precip"].isna()]
nan_precip = len(nc)
nc = df_seoul_analysis[df_seoul_analysis["precip"]==0.0]
no_precip = len(nc)
print(f"강수량 결측치: {nan_precip}개, 0 값: {no_precip}개,NAN 비율: {nan_precip/(len(df_seoul_analysis))*100:.2f}%, 0값 비율: {no_precip/(len(df_seoul_analysis)-nan_precip)*100:.2f}%")
print("====================== 결측치에 대한 강수일 일치확인======================")
n = df_seoul_analysis[df_seoul_analysis["sunshine"].isna()]
print(f"일조시간 결측치에 해당하는 강수일 비율: {n['precip'].notna().mean()*100:.2f}%", f"전체 강수일 비율: {df_seoul_analysis['precip'].notna().mean()*100:.2f}%")

====================== 합계 일조시간 결측치,0 값 비율 확인 ======================
합계 일조시간 결측치: 123개, 0 값: 3094개,NAN 비율: 0.50%, 0값 비율: 12.76%
3094
====================== 일강수량 NAN, 0 값 비율 확인 ======================
강수량 결측치: 14645개, 0 값: 2425개,NAN 비율: 60.11%, 0값 비율: 24.95%
====================== 결측치에 대한 강수일 일치확인======================
일조시간 결측치에 해당하는 강수일 비율: 86.99% 전체 강수일 비율: 39.89%


1961년부터 핵심 항목 결측치 제외

논리적 이상치 확인
평균은 최대,최소 범위 안
기온은 이슬점 초과 불가
상대습도 100 일 때만 일치

값의 이상수치를 아래와 
컬럼	범위	근거
tavg/tmin/tmax	−40 ~ 45 °C	국내 극값(최고 41.0, 최저 −32.6) 바깥 여유
rh	0 ~ 100 %	정의
precip	0 ~ 1000 mm	국내 일강수량 최고 기록 바깥
sunshine	0 ~ 24 hr	정의
wind	0 ~ 30 m/s	일평균 풍속 기준

In [6]:
print("====================== 분석 데이터(서울)======================") 
df_seoul_clean = df_seoul_analysis[
    df_seoul_analysis[CORE].notna().all(axis=1)
    & (df_seoul_analysis["date"] > pd.Timestamp("1960-12-31"))
].copy()

df_err = df_seoul_clean[(df_seoul_clean["tavg"]>df_seoul_clean["tmax"]) | (df_seoul_clean["tavg"]<df_seoul_clean["tmin"])]
print(f"온도 논리 오류 {len(df_err)}건")
display(df_err)

df_err = df_seoul_clean[df_seoul_clean["tavg"]<df_seoul_clean["dew"]]
print(f"이슬점 논리 오류 {len(df_err)}건")
display(df_err)

df_err = df_seoul_clean[(df_seoul_clean["tmax"]>45) | (df_seoul_clean["tmin"]<-40) 
                        | (df_seoul_clean["rh"]>100)| (df_seoul_clean["rh"]<0) 
                        | (df_seoul_clean["rh_min"]>100)| (df_seoul_clean["rh_min"]<0) 
                        | (df_seoul_clean["precip"]>1000)| (df_seoul_clean["precip"]<0) 
                        | (df_seoul_clean["sunshine"]>24) | (df_seoul_clean["sunshine"]<0)
                        | (df_seoul_clean["wind"]>30) | (df_seoul_clean["wind"]<0)
                        ] 
print(f"값 범위 오류 {len(df_err)}건")
display(df_err)


print(f"정제 전 {len(df_seoul_analysis):,}행 → 정제 후 {len(df_seoul_clean):,}행 "
    f"(배제 {len(df_seoul_analysis) - len(df_seoul_clean):,}일)")

df_seoul_clean.info()

df_seoul_clean[CORE].describe() 


====================== 분석 데이터(서울)======================
온도 논리 오류 0건


,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time


이슬점 논리 오류 0건


,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time


값 범위 오류 0건


,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time


정제 전 24,363행 → 정제 후 23,994행 (배제 369일)
<class 'pandas.DataFrame'>
Index: 23994 entries, 366 to 24362
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   stn          23994 non-null  int64         
 1   stn_name     23994 non-null  str           
 2   date         23994 non-null  datetime64[us]
 3   tavg         23994 non-null  float64       
 4   tmin         23994 non-null  float64       
 5   tmax         23994 non-null  float64       
 6   rh           23994 non-null  float64       
 7   rh_min       23994 non-null  float64       
 8   dew          23994 non-null  float64       
 9   vp           23994 non-null  float64       
 10  precip       9566 non-null   float64       
 11  sunshine     23873 non-null  float64       
 12  wind         23985 non-null  float64       
 13  tmax_time    23994 non-null  float64       
 14  rh_min_time  23539 non-null  float64       
dtypes: datetime64[us](1), float64

,tavg,tmin,tmax,rh,rh_min,dew,vp
count,23994.000000,23994.000000,23994.000000,23994.000000,23994.000000,23994.000000,23994.000000
mean,12.544057,8.582867,17.198700,65.156572,44.250925,5.566212,11.823114
std,10.522304,10.618455,10.731672,14.551271,16.530449,11.741708,8.235567
min,-16.400000,-20.200000,-13.600000,17.900000,7.000000,-27.600000,0.700000
25%,3.500000,-0.200000,7.900000,54.800000,32.000000,-3.500000,4.800000
50%,14.000000,9.300000,19.100000,65.500000,43.000000,6.300000,9.600000
75%,21.900000,18.000000,26.500000,75.800000,55.000000,15.800000,18.000000
max,33.700000,30.300000,39.600000,99.800000,97.000000,27.200000,36.200000


## 추풍령 데이터, 전처리

동일한 프로세스를 거쳐서 정제한다.
24362로 하루(1990-01-01)가 결측되었다. 
이슬점에서 역시 60년에  블록 결측을 확인
65,66 년도에 일조시간에 대한 결측을 확인 하였다.
서울과 다르게 1960년도에 최고기온 관측 시간에 결측이 없으나 이미 핵심지표 결측으로 배제되었다.
역시 1960~1962에 걸쳐 블록으로 최저 상대습도 관측 시간 결측이 확인된다.
논리는 동일하게 적용한다.

In [7]:
#원본 데이터 파일 목록
print("====================== 추풍령 ======================")
files = sorted(glob.glob("data/raw/chupungnyeong/chupungnyeong_*.csv"))


df_chupungnyeong = pd.concat((pd.read_csv(f,encoding="cp949") for f in files), ignore_index=True)
print(f"기간: {df_chupungnyeong['일시'].min()} ~ {df_chupungnyeong['일시'].max()}")
print(f"행 {len(df_chupungnyeong)}개")

df_chupungnyeong_analysis = df_chupungnyeong.rename(columns=COLS)[list(COLS.values())].copy()
df_chupungnyeong_analysis["date"] = pd.to_datetime(df_chupungnyeong_analysis["date"])

df_chupungnyeong_analysis.info()

display(df_chupungnyeong_analysis.head())
display(df_chupungnyeong_analysis.loc[12179:12183])
display(df_chupungnyeong_analysis.tail())


df_chupungnyeong_analysis[CORE].describe() 

full_period = pd.date_range(start="1960-01-01", end="2026-09-13", freq='D')

missing_dates = full_period[~full_period.isin(df_chupungnyeong_analysis['date'])]

print(f"결측일 {missing_dates}")

print("====================== 결측치 연간분포 ======================")
na = df_chupungnyeong_analysis.set_index("date")[CORE + AUX].isna()
display(na.resample("YE").sum().loc[lambda d: d.sum(axis=1) > 0])

print("====================== 합계 일조시간 결측치 확인 ======================")
nb = df_chupungnyeong_analysis[(df_chupungnyeong_analysis["date"] >= "1965-01-01") & (df_chupungnyeong_analysis["date"] < "1966-12-31")].set_index("date")[CORE + AUX].isna()
display(nb.resample("ME").sum().loc[lambda d: d.sum(axis=1) > 0])

print("====================== 소수 결측치 ======================")
print("====================== 최저기온 결측치 ======================")
display(df_chupungnyeong_analysis.loc[df_chupungnyeong_analysis["tmin"].isna()][["stn","stn_name","date", "tmin"]])
print("====================== 최고기온 결측치 ======================")
display(df_chupungnyeong_analysis.loc[df_chupungnyeong_analysis["tmax"].isna()][["stn","stn_name","date", "tmax"]])
print("====================== 평균기온 결측치 ======================")
display(df_chupungnyeong_analysis.loc[df_chupungnyeong_analysis["tavg"].isna()][["stn","stn_name","date", "tavg"]])
print("====================== 평균 상대습도 결측치 ======================")
display(df_chupungnyeong_analysis.loc[df_chupungnyeong_analysis["rh"].isna()][["stn","stn_name","date", "rh"]])
print("====================== 최저 상대습도 결측치 ======================")
display(df_chupungnyeong_analysis.loc[df_chupungnyeong_analysis["rh_min"].isna()][["stn","stn_name","date", "rh_min"]])
print("====================== 평균증기압 결측치 ======================")
display(df_chupungnyeong_analysis.loc[df_chupungnyeong_analysis["vp"].isna()][["stn","stn_name","date", "vp"]])
print("====================== 평균 풍속 결측치 ======================")
display(df_chupungnyeong_analysis.loc[df_chupungnyeong_analysis["wind"].isna()][["stn","stn_name","date", "wind"]])

print("====================== 분석 데이터(추풍령)======================") 
df_chupungnyeong_clean = df_chupungnyeong_analysis[
    df_chupungnyeong_analysis[CORE].notna().all(axis=1)
    & (df_chupungnyeong_analysis["date"] > pd.Timestamp("1960-12-31"))
].copy()

df_err = df_chupungnyeong_clean[(df_chupungnyeong_clean["tavg"]>df_chupungnyeong_clean["tmax"]) | (df_chupungnyeong_clean["tavg"]<df_chupungnyeong_clean["tmin"])]
print(f"온도 논리 오류 {len(df_err)}건")
display(df_err)

df_err = df_chupungnyeong_clean[df_chupungnyeong_clean["tavg"]<df_chupungnyeong_clean["dew"]]
print(f"이슬점 논리 오류 {len(df_err)}건")
display(df_err)

df_err = df_chupungnyeong_clean[(df_chupungnyeong_clean["tmax"]>45) | (df_chupungnyeong_clean["tmin"]<-40) 
                        | (df_chupungnyeong_clean["rh"]>100)| (df_chupungnyeong_clean["rh"]<0) 
                        | (df_chupungnyeong_clean["precip"]>1000)| (df_chupungnyeong_clean["precip"]<0) 
                        | (df_chupungnyeong_clean["sunshine"]>24) | (df_chupungnyeong_clean["sunshine"]<0)
                        | (df_chupungnyeong_clean["wind"]>30) | (df_chupungnyeong_clean["wind"]<0)
                        ] 
print(f"값 범위 오류 {len(df_err)}건")
display(df_err)


print(f"정제 전 {len(df_chupungnyeong_analysis):,}행 → 정제 후 {len(df_chupungnyeong_clean):,}행 "
    f"(배제 {len(df_chupungnyeong_analysis) - len(df_chupungnyeong_clean):,}일)")

df_chupungnyeong_clean.info()

df_chupungnyeong_clean[CORE].describe() 



====================== 추풍령 ======================


기간: 1960-01-01 ~ 2026-09-13
행 24362개
<class 'pandas.DataFrame'>
RangeIndex: 24362 entries, 0 to 24361
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   stn          24362 non-null  int64         
 1   stn_name     24362 non-null  str           
 2   date         24362 non-null  datetime64[us]
 3   tavg         24361 non-null  float64       
 4   tmin         24360 non-null  float64       
 5   tmax         24360 non-null  float64       
 6   rh           24360 non-null  float64       
 7   rh_min       24361 non-null  float64       
 8   dew          23995 non-null  float64       
 9   vp           24361 non-null  float64       
 10  precip       9477 non-null   float64       
 11  sunshine     24283 non-null  float64       
 12  wind         24361 non-null  float64       
 13  tmax_time    24356 non-null  float64       
 14  rh_min_time  23540 non-null  float64       
dtypes: datetime64[us](1), float

,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time
0,135,추풍령,1960-01-01,0.4,-2.1,2.9,64.0,53.0,NaN,4.0,NaN,7.5,4.4,1246.0,NaN
1,135,추풍령,1960-01-02,-1.5,-5.4,3.2,71.3,55.0,NaN,4.0,0.0,3.4,2.5,1139.0,NaN
2,135,추풍령,1960-01-03,3.0,-4.4,9.3,72.7,53.0,NaN,5.5,NaN,6.5,0.6,1535.0,NaN
3,135,추풍령,1960-01-04,9.5,1.9,14.4,57.3,37.0,NaN,6.6,NaN,8.5,3.6,1327.0,NaN
4,135,추풍령,1960-01-05,-2.4,-6.1,8.5,50.0,39.0,NaN,2.6,NaN,6.7,6.5,0.0,NaN


,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time
12179,135,추풍령,1993-05-07,18.3,10.0,25.7,59.3,32.0,9.5,11.9,NaN,9.3,1.7,1515.0,1430.0
12180,135,추풍령,1993-05-08,16.5,10.9,21.6,51.0,18.0,5.1,9.2,NaN,9.4,3.0,1630.0,1455.0
12181,135,추풍령,1993-05-09,11.1,9.8,14.3,75.0,51.0,6.8,9.9,21.5,0.0,0.7,1030.0,1125.0
12182,135,추풍령,1993-05-10,15.3,9.0,21.9,56.0,34.0,5.9,9.3,NaN,9.6,3.7,1635.0,1520.0
12183,135,추풍령,1993-05-11,18.1,9.6,25.0,37.3,21.0,2.8,7.5,NaN,7.7,3.6,1418.0,1355.0


,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time
24357,135,추풍령,2026-09-09,18.7,13.1,22.6,69.0,56.0,12.8,14.8,NaN,4.3,2.8,1143.0,1014.0
24358,135,추풍령,2026-09-10,17.1,11.3,22.2,73.1,59.0,12.1,14.2,0.0,2.1,1.9,1519.0,1521.0
24359,135,추풍령,2026-09-11,18.9,12.8,25.1,72.3,46.0,13.4,15.4,NaN,9.0,0.9,1307.0,1350.0
24360,135,추풍령,2026-09-12,19.5,12.5,27.0,70.3,37.0,13.2,15.3,NaN,9.8,0.9,1525.0,1209.0
24361,135,추풍령,2026-09-13,20.5,13.3,27.3,71.6,48.0,14.7,16.7,NaN,11.0,1.2,1357.0,1112.0


...

,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time


값 범위 오류 0건


,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time


정제 전 24,362행 → 정제 후 23,993행 (배제 369일)
<class 'pandas.DataFrame'>
Index: 23993 entries, 366 to 24361
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   stn          23993 non-null  int64         
 1   stn_name     23993 non-null  str           
 2   date         23993 non-null  datetime64[us]
 3   tavg         23993 non-null  float64       
 4   tmin         23993 non-null  float64       
 5   tmax         23993 non-null  float64       
 6   rh           23993 non-null  float64       
 7   rh_min       23993 non-null  float64       
 8   dew          23993 non-null  float64       
 9   vp           23993 non-null  float64       
 10  precip       9335 non-null   float64       
 11  sunshine     23914 non-null  float64       
 12  wind         23992 non-null  float64       
 13  tmax_time    23989 non-null  float64       
 14  rh_min_time  23538 non-null  float64       
dtypes: datetime64[us](1), float64

,tavg,tmin,tmax,rh,rh_min,dew,vp
count,23993.000000,23993.000000,23993.000000,23993.000000,23993.000000,23993.000000,23993.000000
mean,11.821306,6.913212,17.323132,67.980674,44.807206,5.454816,11.564902
std,9.842673,10.023749,10.224875,14.818747,18.059889,11.292332,7.954733
min,-13.900000,-17.800000,-11.100000,21.300000,5.000000,-25.600000,0.900000
25%,3.200000,-1.700000,8.600000,57.300000,31.000000,-4.000000,4.600000
50%,12.900000,6.900000,19.100000,69.800000,43.000000,5.800000,9.300000
75%,20.400000,15.900000,26.000000,79.300000,57.000000,15.500000,17.700000
max,30.000000,26.900000,36.800000,98.300000,97.000000,25.800000,33.200000


## 대구 데이터, 전처리

결측일 없음
동일한 1960년 이슬점 블록 결측
동일한 패턴의 일조시간 결측
동일한 패턴의 최저습도관측 시간 결측
동일한 로직으로 처리함

2025-01-09 이슬점 논리 오류 1건

In [8]:
#원본 데이터 파일 목록
print("====================== 대구 ======================")
files = sorted(glob.glob("data/raw/daegu/daegu*.csv"))


df_daegu = pd.concat((pd.read_csv(f,encoding="cp949") for f in files), ignore_index=True)
print(f"기간: {df_daegu['일시'].min()} ~ {df_daegu['일시'].max()}")
print(f"행 {len(df_daegu)}개")

df_daegu_analysis = df_daegu.rename(columns=COLS)[list(COLS.values())].copy()
df_daegu_analysis["date"] = pd.to_datetime(df_daegu_analysis["date"])

df_daegu_analysis.info()

display(df_daegu_analysis.head())
display(df_daegu_analysis.loc[12179:12183])
display(df_daegu_analysis.tail())


df_daegu_analysis[CORE].describe() 

full_period = pd.date_range(start="1960-01-01", end="2026-09-13", freq='D')

missing_dates = full_period[~full_period.isin(df_daegu_analysis['date'])]

print("====================== 결측치 연간분포(대구) ======================")
na = df_daegu_analysis.set_index("date")[CORE + AUX].isna()
display(na.resample("YE").sum().loc[lambda d: d.sum(axis=1) > 0])

print("====================== 합계 일조시간 결측치 확인(대구) ======================")
nb = df_daegu_analysis[(df_daegu_analysis["date"] >= "1965-01-01") & (df_daegu_analysis["date"] < "1966-12-31")].set_index("date")[CORE + AUX].isna()
display(nb.resample("ME").sum().loc[lambda d: d.sum(axis=1) > 0])

print("====================== 소수 결측치(대구) ======================")
print("====================== 최저기온 결측치 ======================")
display(df_daegu_analysis.loc[df_daegu_analysis["tmin"].isna()][["stn","stn_name","date", "tmin"]])
print("====================== 최고기온 결측치 ======================")
display(df_daegu_analysis.loc[df_daegu_analysis["tmax"].isna()][["stn","stn_name","date", "tmax"]])
print("====================== 평균기온 결측치 ======================")
display(df_daegu_analysis.loc[df_daegu_analysis["tavg"].isna()][["stn","stn_name","date", "tavg"]])
print("====================== 평균 상대습도 결측치 ======================")
display(df_daegu_analysis.loc[df_daegu_analysis["rh"].isna()][["stn","stn_name","date", "rh"]])
print("====================== 최저 상대습도 결측치 ======================")
display(df_daegu_analysis.loc[df_daegu_analysis["rh_min"].isna()][["stn","stn_name","date", "rh_min"]])

print("====================== 평균증기압 결측치 ======================")
display(df_daegu_analysis.loc[df_daegu_analysis["vp"].isna()][["stn","stn_name","date", "vp"]])
print("====================== 평균 풍속 결측치 ======================")
display(df_daegu_analysis.loc[df_daegu_analysis["wind"].isna()][["stn","stn_name","date", "wind"]])

print("====================== 분석 데이터(대구)======================") 
df_daegu_clean = df_daegu_analysis[
    df_daegu_analysis[CORE].notna().all(axis=1)
    & (df_daegu_analysis["date"] > pd.Timestamp("1960-12-31"))
].copy()

df_err = df_daegu_clean[(df_daegu_clean["tavg"]>df_daegu_clean["tmax"]) | (df_daegu_clean["tavg"]<df_daegu_clean["tmin"])]
print(f"온도 논리 오류 {len(df_err)}건")
display(df_err)

df_err = df_daegu_clean[df_daegu_clean["tavg"]<df_daegu_clean["dew"]]
print(f"이슬점 논리 오류 {len(df_err)}건")
display(df_err)

df_err = df_daegu_clean[(df_daegu_clean["tmax"]>45) | (df_daegu_clean["tmin"]<-40) 
                        | (df_daegu_clean["rh"]>100)| (df_daegu_clean["rh"]<0) 
                        | (df_daegu_clean["precip"]>1000)| (df_daegu_clean["precip"]<0) 
                        | (df_daegu_clean["sunshine"]>24) | (df_daegu_clean["sunshine"]<0)
                        | (df_daegu_clean["wind"]>30) | (df_daegu_clean["wind"]<0)
                        ] 
print(f"값 범위 오류 {len(df_err)}건")
display(df_err)
#이상치 배제
df_daegu_clean = df_daegu_clean[df_daegu_clean["tavg"]>=df_daegu_clean["dew"]]

print(f"정제 전 {len(df_daegu_analysis):,}행 → 정제 후 {len(df_daegu_clean):,}행 "
    f"(배제 {len(df_daegu_analysis) - len(df_daegu_clean):,}일)")

df_daegu_clean.info()

df_daegu_clean[CORE].describe() 

====================== 대구 ======================
기간: 1960-01-01 ~ 2026-09-13
행 24363개
<class 'pandas.DataFrame'>
RangeIndex: 24363 entries, 0 to 24362
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   stn          24363 non-null  int64         
 1   stn_name     24363 non-null  str           
 2   date         24363 non-null  datetime64[us]
 3   tavg         24362 non-null  float64       
 4   tmin         24362 non-null  float64       
 5   tmax         24360 non-null  float64       
 6   rh           24361 non-null  float64       
 7   rh_min       24361 non-null  float64       
 8   dew          23995 non-null  float64       
 9   vp           24361 non-null  float64       
 10  precip       8329 non-null   float64       
 11  sunshine     24295 non-null  float64       
 12  wind         24357 non-null  float64       
 13  tmax_time    24358 non-null  float64       
 14  rh_min_time  23540 non-null

,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time
0,143,대구,1960-01-01,1.6,-0.8,5.7,64.7,50.0,NaN,4.3,NaN,8.4,4.1,1337.0,NaN
1,143,대구,1960-01-02,0.7,-3.0,5.4,65.3,48.0,NaN,4.1,NaN,7.1,2.5,1528.0,NaN
2,143,대구,1960-01-03,3.4,-3.0,11.1,72.0,45.0,NaN,5.6,NaN,8.7,1.5,1542.0,NaN
3,143,대구,1960-01-04,10.5,4.0,16.3,60.0,39.0,NaN,7.3,NaN,8.6,2.8,1434.0,NaN
4,143,대구,1960-01-05,0.0,-4.0,7.6,49.0,42.0,NaN,3.0,NaN,7.2,8.3,NaN,NaN


,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time
12179,143,대구,1993-05-06,17.5,15.1,22.1,83.8,59.0,14.7,16.7,2.3,3.0,1.6,1707.0,1642.0
12180,143,대구,1993-05-07,18.7,13.1,25.2,72.3,47.0,13.1,15.2,NaN,9.7,3.9,1553.0,1535.0
12181,143,대구,1993-05-08,19.9,14.1,26.5,57.3,27.0,9.9,12.6,NaN,11.3,2.5,1432.0,1718.0
12182,143,대구,1993-05-09,13.8,11.1,16.7,85.0,54.0,11.3,13.4,26.4,0.0,1.7,1225.0,0.0
12183,143,대구,1993-05-10,16.5,10.6,24.5,71.3,41.0,10.6,12.8,1.9,10.4,2.8,1622.0,1552.0


,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time
24358,143,대구,2026-09-09,21.2,17.5,24.2,55.1,44.0,11.7,13.9,NaN,3.8,2.6,1036.0,2057.0
24359,143,대구,2026-09-10,20.1,14.3,24.8,59.1,41.0,11.4,13.5,NaN,6.5,2.1,1222.0,1120.0
24360,143,대구,2026-09-11,22.7,18.4,28.4,59.0,36.0,13.8,15.8,0.1,8.8,1.2,1611.0,1508.0
24361,143,대구,2026-09-12,23.2,17.2,29.9,62.4,20.0,14.4,16.6,NaN,11.7,1.7,1524.0,1213.0
24362,143,대구,2026-09-13,24.2,17.6,31.3,62.4,31.0,15.7,17.8,NaN,10.4,1.1,1456.0,1558.0


====================== 결측치 연간분포(대구) ======================


...

,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time
23750,143,대구,2025-01-09,-6.1,-10.1,-1.6,42.4,30.0,-4.6,1.7,NaN,6.2,5.1,1.0,632.0


값 범위 오류 0건


,stn,stn_name,date,tavg,tmin,tmax,rh,rh_min,dew,vp,precip,sunshine,wind,tmax_time,rh_min_time


정제 전 24,363행 → 정제 후 23,992행 (배제 371일)
<class 'pandas.DataFrame'>
Index: 23992 entries, 366 to 24362
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   stn          23992 non-null  int64         
 1   stn_name     23992 non-null  str           
 2   date         23992 non-null  datetime64[us]
 3   tavg         23992 non-null  float64       
 4   tmin         23992 non-null  float64       
 5   tmax         23992 non-null  float64       
 6   rh           23992 non-null  float64       
 7   rh_min       23992 non-null  float64       
 8   dew          23992 non-null  float64       
 9   vp           23992 non-null  float64       
 10  precip       8212 non-null   float64       
 11  sunshine     23924 non-null  float64       
 12  wind         23987 non-null  float64       
 13  tmax_time    23992 non-null  float64       
 14  rh_min_time  23537 non-null  float64       
dtypes: datetime64[us](1), float64

,tavg,tmin,tmax,rh,rh_min,dew,vp
count,23992.000000,23992.000000,23992.000000,23992.000000,23992.000000,23992.000000,23992.00000
mean,14.016518,9.332323,19.477313,62.849025,40.717018,6.218094,12.11488
std,9.682470,9.903237,9.892211,15.340664,17.001916,11.272834,8.12562
min,-10.400000,-15.400000,-7.600000,13.500000,4.000000,-25.600000,0.80000
25%,5.500000,0.700000,11.000000,52.000000,28.000000,-2.900000,5.00000
50%,14.950000,9.600000,20.800000,64.000000,39.000000,7.000000,10.10000
75%,22.300000,18.100000,27.700000,73.900000,51.000000,16.000000,18.30000
max,33.100000,28.800000,39.600000,99.800000,97.000000,26.900000,35.30000


## 처리 완료
모두 1960년에 이슬점 블록 결측이 확인되어 제외 하였으며,
각데이터의 핵심 항목 결측치와 결측일(추풍령 1990-01-01) 논리적 오류가 확인된 일자(대구 2025-01-09)를 제외하였다.
일강수량은 결측값이 비강수일에 해당 되어 정확한 결측일을 알기 어려움에 따라 결측일을 배제 하기 어려운 한계가 있어 결측여부를 특정하지 않기로 하였다.
그외 보조지표( "합계 일조시간(hr)": "sunshine", "평균 풍속(m/s)": "wind")를 이용시 이용시에는 해당항목 결측일을 후에 배제하기로 한다.
특히 1965년과 1966년에 일조시간에 대한 동일한 패턴의 블록 결측치가 관측 된것을 확인하였다.
1960년의 이슬점과 1965년~1966년의 일조량에는 측정 기준이 바뀌거나 신설되는 등 이유가 있을 것으로 보이나, 현재 분석의 핵심 주제가 아니여서 더 이상 자세한 분석은 진행 하지 않는다.

In [9]:
clean = {
    "서울": df_seoul_clean,
    "대구": df_daegu_clean,
    "추풍령": df_chupungnyeong_clean,
}

final = pd.concat(
    [d.assign(station=name) for name, d in clean.items()],
    ignore_index=True
).sort_values(["station", "date"]).reset_index(drop=True)

# 지점별 정제 결과 요약 — 일수가 다른 것이 정상이다
for name, d in clean.items():
    print(f"{name:4s} {len(d):,}일  {d.date.min():%Y-%m-%d} ~ {d.date.max():%Y-%m-%d}  "
          f"핵심결측 {d[CORE].isna().sum().sum()}건")

common = set.intersection(*(set(d.date) for d in clean.values()))
print(f"\n3지점 공통 유효일 {len(common):,}일 (Q2 지역비교에 사용)")

final.to_csv("data/processed/asos_clean.csv", index=False, encoding="utf-8")
print(f"\n저장: {len(final):,}행")

서울   23,994일  1961-01-01 ~ 2026-09-13  핵심결측 0건
대구   23,992일  1961-01-01 ~ 2026-09-13  핵심결측 0건
추풍령  23,993일  1961-01-01 ~ 2026-09-13  핵심결측 0건

3지점 공통 유효일 23,987일 (Q2 지역비교에 사용)

저장: 71,979행
